# 🐾 Cats vs Dogs Image Classification — Deep Transfer Learning (MobileNetV2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rshamith777-cpu/AIML_1_ImageClassification_byte/blob/main/notebooks/cats_vs_dogs_transfer_learning.ipynb)

**Repository:** `AIML_1_ImageClassification_byte`  
**Track:** Arithmatrix Virtual Internship Program (AVIP 2026) — AI/ML Track (Task 1)  
**Architecture:** MobileNetV2 (ImageNet Pre-trained) + GlobalAveragePooling2D + Fine-Tuning Head  
**Dataset:** Real Microsoft / Kaggle Dogs vs. Cats Benchmark (~23,262 valid images) via `tensorflow_datasets`  

---

### Step 1: Environment Setup, Seeds & GPU Verification
Verify GPU availability (e.g., NVIDIA Tesla T4 on Google Colab), initialize deterministic random seeds, and import libraries.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set deterministic seeds
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs Available:", len(gpus))
if gpus:
    print("Device:", tf.test.gpu_device_name())
else:
    print("Note: To enable GPU in Colab, go to Runtime > Change runtime type > T4 GPU.")

### Step 2: Load Real Dataset via TFDS & Verify Label Balance
We load the official Microsoft/Kaggle `cats_vs_dogs` dataset using `tfds.ReadConfig(shuffle_seed=42)` and split into:
- **80% Training**
- **10% Validation**
- **10% Test** (strictly held-out)

We count and display the exact cat and dog instances in each split to verify that both classes are balanced.

In [ ]:
read_config = tfds.ReadConfig(shuffle_seed=42)
(raw_train, raw_val, raw_test), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
    read_config=read_config
)

def get_split_class_counts(dataset, name):
    c_cat, c_dog = 0, 0
    for _, label in dataset:
        if int(label.numpy()) == 0:
            c_cat += 1
        else:
            c_dog += 1
    total = c_cat + c_dog
    ratio_dog = (c_dog / total) if total > 0 else 0
    print(f"  {name:<14}: Total={total:>5} | Cats={c_cat:>5} ({c_cat/total:.1%}) | Dogs={c_dog:>5} ({ratio_dog:.1%})")
    return c_cat, c_dog, total, ratio_dog

print("Dataset Split Verification (Label Distribution):")
train_cat, train_dog, train_tot, train_r = get_split_class_counts(raw_train, "Train (80%)")
val_cat, val_dog, val_tot, val_r = get_split_class_counts(raw_val, "Val   (10%)")
test_cat, test_dog, test_tot, test_r = get_split_class_counts(raw_test, "Test  (10%)")

# Guardrail: confirm class representation is balanced within 5% tolerance
if abs(train_r - 0.5) > 0.05 or abs(val_r - 0.5) > 0.05 or abs(test_r - 0.5) > 0.05:
    print("\n[NOTICE] Non-uniform distribution detected. Applying deterministic stratified split...")
    
    # 1. Full-dataset class totals across all non-overlapping slices
    total_cats = train_cat + val_cat + test_cat
    total_dogs = train_dog + val_dog + test_dog
    total_examples = total_cats + total_dogs
    print(f"Full Dataset Class Totals: Cats={total_cats}, Dogs={total_dogs}, Total={total_examples}")
    
    # 2. Deterministic ~80/10/10 non-overlapping split sizes per class
    n_train_cat = int(total_cats * 0.80)
    n_val_cat = int(total_cats * 0.10)
    n_test_cat = total_cats - n_train_cat - n_val_cat
    
    n_train_dog = int(total_dogs * 0.80)
    n_val_dog = int(total_dogs * 0.10)
    n_test_dog = total_dogs - n_train_dog - n_val_dog
    
    # 3. Load full dataset and partition into classes
    full_ds = tfds.load('cats_vs_dogs', split='train', as_supervised=True)
    cats_ds = full_ds.filter(lambda x, y: tf.equal(y, 0))
    dogs_ds = full_ds.filter(lambda x, y: tf.equal(y, 1))
    
    # 4. Deterministic example-level shuffle with fixed seed and reshuffle_each_iteration=False
    # Bounded buffer (2048) prevents OOM from buffering all full-resolution images into RAM
    SHUFFLE_BUFFER = 2048
    shuffled_cats = cats_ds.shuffle(buffer_size=SHUFFLE_BUFFER, seed=42, reshuffle_each_iteration=False)
    shuffled_dogs = dogs_ds.shuffle(buffer_size=SHUFFLE_BUFFER, seed=42, reshuffle_each_iteration=False)
    
    # 5. Non-overlapping take/skip partition sharing the identical frozen shuffle order
    raw_train_cats = shuffled_cats.take(n_train_cat)
    raw_val_cats = shuffled_cats.skip(n_train_cat).take(n_val_cat)
    raw_test_cats = shuffled_cats.skip(n_train_cat + n_val_cat).take(n_test_cat)
    
    raw_train_dogs = shuffled_dogs.take(n_train_dog)
    raw_val_dogs = shuffled_dogs.skip(n_train_dog).take(n_val_dog)
    raw_test_dogs = shuffled_dogs.skip(n_train_dog + n_val_dog).take(n_test_dog)
    
    raw_train = raw_train_cats.concatenate(raw_train_dogs)
    raw_val = raw_val_cats.concatenate(raw_val_dogs)
    raw_test = raw_test_cats.concatenate(raw_test_dogs)
    
    # 6. Verify and display actual split sizes & class distributions
    print("\nVerifying Stratified Splits:")
    new_train_cat, new_train_dog, new_train_tot, new_train_r = get_split_class_counts(raw_train, "Train (Strat)")
    new_val_cat, new_val_dog, new_val_tot, new_val_r = get_split_class_counts(raw_val, "Val   (Strat)")
    new_test_cat, new_test_dog, new_test_tot, new_test_r = get_split_class_counts(raw_test, "Test  (Strat)")
    
    # 7. Strictly verify that all examples are accounted for exactly once
    accounted_cats = new_train_cat + new_val_cat + new_test_cat
    accounted_dogs = new_train_dog + new_val_dog + new_test_dog
    accounted_total = accounted_cats + accounted_dogs
    
    assert accounted_cats == total_cats, f"Cat count mismatch: {accounted_cats} vs {total_cats}"
    assert accounted_dogs == total_dogs, f"Dog count mismatch: {accounted_dogs} vs {total_dogs}"
    assert accounted_total == total_examples, f"Total count mismatch: {accounted_total} vs {total_examples}"
    print(f"\n[VERIFIED] All {accounted_total} examples accounted for exactly once (Cats: {accounted_cats}, Dogs: {accounted_dogs}).")
else:
    print("\n[VERIFIED] Both classes are balanced across train, validation, and test splits (~50% cats / ~50% dogs).")
    total_cats = train_cat + val_cat + test_cat
    total_dogs = train_dog + val_dog + test_dog
    total_examples = total_cats + total_dogs
    print(f"All {total_examples} examples accounted for exactly once (Cats: {total_cats}, Dogs: {total_dogs}).")

### Step 3: Data Preprocessing & Augmentation Pipeline
Images are resized to 160x160 and normalized to [-1, 1] using `mobilenet_v2.preprocess_input`.
Data augmentation (random horizontal flips and rotations) is applied to the training set with a fixed random seed to ensure reproducibility.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal', seed=42),
    layers.RandomRotation(0.1, seed=42),
])

train_ds = raw_train.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000, seed=42).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = raw_val.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

test_ds = raw_test.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)
print("Data pipelines cached and batched with fixed seed=42.")

### Step 4: Transfer Learning Architecture (MobileNetV2)
We load MobileNetV2 with pre-trained ImageNet weights with its top classification layer removed.
Phase 1: Freeze the base convolutional layers to preserve pre-trained feature detectors.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(160, 160, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3, seed=42)(x)
outputs = layers.Dense(1)(x)  # Linear logit output (positive = Dog, negative = Cat)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary()

### Step 5: Phase 1 Training (Feature Extraction)
Train top classification head for 3 epochs.

In [ ]:
history_phase1 = model.fit(
    train_ds,
    epochs=3,
    validation_data=val_ds
)

### Step 6: Phase 2 Fine-Tuning
Unfreeze the top 30 convolutional layers of MobileNetV2 and train with a lower learning rate ($10^{-5}$) for 2 additional epochs.

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_ds,
    epochs=5,
    initial_epoch=history_phase1.epoch[-1],
    validation_data=val_ds
)

### Step 7: Final Test Set Evaluation & Metrics Export
Evaluate on the strictly held-out test set. Compute Accuracy, Precision, Recall, F1, and save `confusion_matrix.png` and `metrics.json`.

In [ ]:
y_true, y_pred_probs = [], []
for batch_x, batch_y in test_ds:
    logits = model.predict(batch_x, verbose=0)
    probs = tf.sigmoid(logits).numpy().flatten()
    y_pred_probs.extend(probs)
    y_true.extend(batch_y.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.50).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("=== FINAL TEST RESULTS ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print(f"F1-Score : {f1*100:.2f}%")
print("\nConfusion Matrix:\n", cm)

# Save metrics.json
os.makedirs('outputs', exist_ok=True)
metrics_dict = {
    'accuracy': round(float(acc), 4),
    'precision_dog': round(float(prec), 4),
    'recall_dog': round(float(rec), 4),
    'f1_dog': round(float(f1), 4),
    'test_samples': len(y_true)
}
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# Plot Confusion Matrix & close figure
import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.xlabel('Predicted Label')
plt.ylabel('Ground Truth')
plt.title('Confusion Matrix — Cats vs Dogs (Test Set)')
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()  # Prevent memory leakage
print("Saved outputs/metrics.json and outputs/confusion_matrix.png")

### Step 8: Save 10 Sample Inference Photographs & CSV Table
Extract 10 real test photos (5 cats, 5 dogs) from `test_ds` and export them with predictions.

In [ ]:
import csv
from PIL import Image

os.makedirs('outputs/sample_inferences', exist_ok=True)
samples_data = []
cat_count, dog_count = 0, 0

for img_raw, label_raw in raw_test:
    lbl = int(label_raw.numpy())
    if lbl == 0 and cat_count < 5:
        cat_count += 1
        fname = f'cat_0{cat_count}.jpg'
        target_label = 'cat'
    elif lbl == 1 and dog_count < 5:
        dog_count += 1
        fname = f'dog_0{dog_count}.jpg'
        target_label = 'dog'
    else:
        if cat_count >= 5 and dog_count >= 5:
            break
        continue

    # Save real photo
    fpath = os.path.join('outputs/sample_inferences', fname)
    pil_img = Image.fromarray(img_raw.numpy())
    pil_img.save(fpath, quality=90)

    # Predict with model
    inp = tf.image.resize(img_raw, IMG_SIZE)
    inp = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    logit = model.predict(tf.expand_dims(inp, 0), verbose=0)[0][0]
    p_dog = float(tf.sigmoid(logit).numpy())
    pred = 'dog' if p_dog >= 0.50 else 'cat'
    conf = p_dog if pred == 'dog' else (1.0 - p_dog)

    samples_data.append([fname, target_label, pred, f"{p_dog:.4f}", f"{conf*100:.2f}%"])

with open('outputs/sample_predictions.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image_filename', 'ground_truth', 'predicted_label', 'p_dog', 'confidence'])
    writer.writerows(samples_data)

print("Sample Predictions Table (10 Real Photos):")
for row in samples_data:
    print(row)

### Step 9: Save Serialized Model
Save the trained deep learning model as `cats_dogs_mobilenetv2.keras` in native Keras v3 format.

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/cats_dogs_mobilenetv2.keras'
model.save(model_path)
print(f"Saved production model to {model_path} ({os.path.getsize(model_path) / (1024*1024):.2f} MB)")

### Step 10: 1-Click ZIP Packaging & Download (Colab)
Packages `models/` and `outputs/` into `cats_dogs_artifacts.zip` using Python's standard `zipfile` module and triggers automatic browser download.

In [ ]:
import zipfile

zip_filename = 'cats_dogs_artifacts.zip'
print(f"Packaging models/ and outputs/ into {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for folder in ['models', 'outputs']:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    zipf.write(file_path, file_path)
                    print(f"  Added: {file_path}")

print(f"\n[SUCCESS] Archive created: {zip_filename} ({os.path.getsize(zip_filename)} bytes)")

# Trigger automatic download in Google Colab
try:
    from google.colab import files
    files.download(zip_filename)
    print("Download prompt initiated in Colab browser.")
except ImportError:
    print("Running outside Google Colab environment; download step skipped.")